[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juliopez/Taller-Fundamentos-Data-Science-Python/blob/main/Machine_Learning_2026/03_Notebooks/NB06_Transfer_Learning_Fine_Tuning_Evaluacion_CNN.ipynb)
# NB06 — Evaluación de modelos preentrenados con imágenes

## Propósito
Experimentar con **MobileNetV2** (entrenada en ImageNet) **sin entrenar ni ajustar pesos**. Primero analizaremos un conjunto común de **20 imágenes CIFAR-10 de GitHub** (dos por categoría); después **deberás** repetir el experimento con tu propio dataset del NB05 y probar una imagen individual subida a Colab.

**Objetivos:** observar predicciones y probabilidades, distinguir aciertos/errores con criterio semántico y reconocer limitaciones de resolución y de clases. **No se realiza Fine-Tuning.**

**Importante:** CIFAR-10 tiene diez categorías; ImageNet tiene mil clases que no corresponden necesariamente a las mismas etiquetas. Una coincidencia textual exacta no es una medida adecuada de acierto.

## 1. Preparar el entorno

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from pathlib import Path
from PIL import Image
from IPython.display import display
import urllib.request

print("TensorFlow:", tf.__version__)

## 2. Descargar el conjunto de ejemplo desde GitHub

El notebook descarga directamente las **20 imágenes PNG**, organizadas en diez carpetas por categoría, desde el repositorio del curso. No necesitas descargar ZIP ni conectar Google Drive para esta primera parte.

Fuente: [diccionario de datos NB06_CIFAR10](https://github.com/juliopez/Taller-Fundamentos-Data-Science-Python/blob/main/Machine_Learning_2026/02_Datasets/NB06_CIFAR10/README.md).

In [ ]:
BASE_RAW = "https://raw.githubusercontent.com/juliopez/Taller-Fundamentos-Data-Science-Python/main/Machine_Learning_2026/02_Datasets/NB06_CIFAR10"
CLASES_CIFAR10 = ["avion", "automovil", "pajaro", "gato", "ciervo", "perro", "rana", "caballo", "barco", "camion"]
DESTINO = Path("/content/NB06_CIFAR10")
registros = []
for clase in CLASES_CIFAR10:
    carpeta = DESTINO / clase
    carpeta.mkdir(parents=True, exist_ok=True)
    for numero in (1, 2):
        nombre = f"{clase}_{numero:02d}.png"
        ruta = carpeta / nombre
        if not ruta.exists():
            urllib.request.urlretrieve(f"{BASE_RAW}/{clase}/{nombre}", ruta)
        registros.append({"categoria_referencia": clase, "ruta": str(ruta), "imagen": nombre})
muestra = pd.DataFrame(registros)
assert len(muestra) == 20 and muestra["categoria_referencia"].nunique() == 10
print("Imágenes descargadas:", len(muestra))
display(muestra)

## 3. Observar el dataset antes de predecir

Revisa las imágenes y sus categorías originales. CIFAR-10 contiene imágenes de **32 × 32 píxeles**: al ampliarlas para MobileNetV2 no se recupera información visual perdida.

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(4, 5, figsize=(14, 10))
for ax, fila in zip(axes.flat, muestra.itertuples(index=False)):
    with Image.open(fila.ruta) as im:
        ax.imshow(im.convert("RGB").resize((128, 128)))
    ax.set_title(fila.categoria_referencia + "\n" + fila.imagen, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Conocer y cargar MobileNetV2 preentrenada

### ¿Qué es MobileNetV2?

**MobileNetV2** es una red neuronal convolucional presentada por investigadores de Google en **2018**. Fue diseñada para clasificar imágenes utilizando relativamente pocos recursos computacionales, por lo que resulta especialmente interesante para teléfonos móviles y dispositivos con capacidad limitada. Su arquitectura emplea bloques residuales invertidos y cuellos de botella lineales para procesar imágenes de forma eficiente.

### ¿Por qué utilizamos un modelo preentrenado?

Entrenar una CNN desde cero exige muchas imágenes etiquetadas, tiempo y capacidad de cómputo. En cambio, un **modelo preentrenado** ya aprendió patrones visuales a partir de un conjunto amplio de imágenes. Podemos reutilizarlo para hacer predicciones (**inferencia**) y analizar sus resultados sin volver a entrenarlo. Más adelante, técnicas como *Transfer Learning* permiten adaptar parte de ese conocimiento a categorías nuevas.

En esta práctica cargaremos MobileNetV2 con pesos entrenados en **ImageNet**, que permiten predecir entre **1.000 clases**. Esas clases no coinciden necesariamente con las diez categorías de CIFAR-10.

### ¿Cómo se comparan los modelos de clasificación?

Entre **2010 y 2017**, el desafío anual **ImageNet Large Scale Visual Recognition Challenge (ILSVRC)** impulsó grandes avances en clasificación de imágenes. Algunas arquitecturas históricas fueron **AlexNet (2012)**, **VGG (2014)** y **ResNet (2015)**; **MobileNetV2 (2018)** surgió después, con énfasis en la eficiencia.

Hoy no existe un único «ranking anual» universal: los modelos se comparan mediante distintos conjuntos de prueba y métricas, como **exactitud Top-1**, **exactitud Top-5**, cantidad de parámetros, latencia y consumo de recursos. Un modelo más preciso no siempre es el más conveniente para una aplicación concreta.

**En NB06 utilizaremos MobileNetV2 como ejemplo de modelo eficiente y accesible**, no porque sea necesariamente el mejor modelo disponible.

**Lecturas de referencia:** [artículo MobileNetV2 (2018)](https://arxiv.org/abs/1801.04381) · [historia de ILSVRC](https://www.image-net.org/challenges/LSVRC/) · [documentación de MobileNetV2 en Keras](https://keras.io/api/applications/mobilenet/).

La siguiente celda descarga los pesos preentrenados la primera vez que se ejecuta. **No entrenaremos ni modificaremos los pesos del modelo.**

In [ ]:
mobilenet = keras.applications.MobileNetV2(weights="imagenet", include_top=True)
print("Parámetros del modelo:", mobilenet.count_params())

## 5. Ejecutar las predicciones

Cada imagen se redimensiona a 224 × 224 y se preprocesa con la función específica de MobileNetV2. Mostraremos las **tres etiquetas ImageNet más probables** y la probabilidad de la primera. Estas probabilidades no equivalen a una garantía de acierto.

In [ ]:
def inferir_mobilenet(ruta):
    with Image.open(ruta) as im:
        x = np.asarray(im.convert("RGB").resize((224, 224)), dtype=np.float32)[None, ...]
    x = keras.applications.mobilenet_v2.preprocess_input(x)
    probabilidades = mobilenet.predict(x, verbose=0)
    return keras.applications.mobilenet_v2.decode_predictions(probabilidades, top=3)[0]

def predecir_tabla(datos):
    filas = []
    for fila in datos.itertuples(index=False):
        preds = inferir_mobilenet(fila.ruta)
        filas.append({
            "categoria_referencia": fila.categoria_referencia,
            "imagen": fila.imagen,
            "prediccion_top1": preds[0][1],
            "probabilidad_top1": float(preds[0][2]),
            "top3": " | ".join(f"{etiqueta} ({prob:.1%})" for _, etiqueta, prob in preds),
        })
    return pd.DataFrame(filas)

resultados = predecir_tabla(muestra)
display(resultados)

## 6. Analizar aciertos y errores

Observa las imágenes y compara su categoría de referencia con la etiqueta ImageNet. **No uses una igualdad automática de nombres:** ImageNet puede emplear categorías más específicas o diferentes. Clasifica manualmente cada resultado como **coherente**, **parcialmente relacionado** o **incorrecto**, justificando tu criterio.

La siguiente tabla permite anotar tu evaluación sin modificar las predicciones.

In [ ]:
analisis = resultados[["categoria_referencia", "imagen", "prediccion_top1", "probabilidad_top1"]].copy()
analisis["evaluacion"] = ""  # coherente / parcialmente relacionado / incorrecto
analisis["justificacion"] = ""
display(analisis)
print("Completa las dos últimas columnas según tu análisis; no se calcula una exactitud automática.")

## 7. Actividad autónoma — dataset propio del NB05

Ahora utiliza **tu propio dataset del NB05** para repetir el experimento con imágenes diferentes a las de CIFAR-10. Conecta Google Drive y configura la carpeta que utilizaste en NB05, organizada en subcarpetas por categoría.

Se seleccionarán **hasta dos imágenes por categoría**, sin exigir un mínimo fijo de categorías o imágenes. Observa las predicciones y contrástalas con las etiquetas de tus carpetas.

Antes de ejecutar, cambia `DATASET_DIR` por la ruta real de tu dataset.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATASET_DIR = Path("/content/drive/MyDrive/MachineLearning2026/dataset")  # AJUSTAR a tu NB05
assert DATASET_DIR.is_dir(), f"No existe la carpeta: {DATASET_DIR}"
EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
propios = []
for carpeta in sorted(p for p in DATASET_DIR.iterdir() if p.is_dir()):
    imagenes = sorted(p for p in carpeta.rglob("*") if p.is_file() and p.suffix.lower() in EXT)
    for ruta in imagenes[:2]:
        propios.append({"categoria_referencia": carpeta.name, "ruta": str(ruta), "imagen": ruta.name})
assert propios, "No se encontraron imágenes compatibles en las carpetas de clases"
muestra_propia = pd.DataFrame(propios)
resultados_propios = predecir_tabla(muestra_propia)
print("Imágenes propias evaluadas:", len(muestra_propia))
display(resultados_propios)

analisis_propio = resultados_propios[["categoria_referencia", "imagen", "prediccion_top1", "probabilidad_top1"]].copy()
analisis_propio["evaluacion"] = ""  # coherente / parcialmente relacionado / incorrecto
analisis_propio["justificacion"] = ""
display(analisis_propio)

## 8. Subir una imagen individual desde tu computador

Esta actividad permite probar MobileNetV2 con **una imagen nueva**, sin agregarla al dataset ni volver a entrenar el modelo.

1. Ejecuta la siguiente celda y selecciona una imagen JPG, PNG, BMP o WEBP.
2. Observa la imagen y sus **tres predicciones ImageNet más probables**.
3. Explica en una celda Markdown si la clasificación tiene sentido y por qué una probabilidad alta no garantiza un acierto.

In [ ]:
from google.colab import files

archivos_subidos = files.upload()
assert archivos_subidos, "No se seleccionó ninguna imagen"
nombre_archivo = next(iter(archivos_subidos))
ruta_individual = Path("/content") / nombre_archivo
with Image.open(ruta_individual) as im:
    imagen_individual = im.convert("RGB")
    display(imagen_individual.copy().resize((224, 224)))

predicciones_individuales = inferir_mobilenet(ruta_individual)
tabla_individual = pd.DataFrame([
    {"posicion": i, "etiqueta_imagenet": etiqueta, "probabilidad": float(prob)}
    for i, (_, etiqueta, prob) in enumerate(predicciones_individuales, start=1)
])
print("Imagen analizada:", nombre_archivo)
display(tabla_individual)

## 9. Preguntas para responder en celdas Markdown

**A.** Identifica dos imágenes de CIFAR-10: ¿qué predijo MobileNetV2 y con qué probabilidad? ¿Son coherentes las predicciones?

**B.** Describe un caso en que la etiqueta ImageNet sea más específica o distinta de la categoría CIFAR-10.

**C.** ¿Cómo influye la resolución original de 32 × 32 píxeles? ¿Por qué redimensionar no recupera información perdida?

**D.** **Dataset propio del NB05 (obligatorio):** identifica dos predicciones, evalúa su coherencia con las categorías reales y compara con el conjunto CIFAR-10.

**E.** **Imagen individual:** indica qué imagen subiste, cuáles fueron las predicciones y si el resultado tiene sentido.

**F.** ¿Por qué una probabilidad alta no garantiza un acierto? ¿Cómo podría ayudar el *Transfer Learning* a adaptar un modelo a tus categorías, sin implementarlo?

## 10. Cierre y continuidad

En este NB06 has evaluado **MobileNetV2 preentrenada** con un conjunto común, **tu dataset del NB05** y una **imagen individual subida a Colab**, sin reentrenar pesos.

**NB07:** preparación e interoperabilidad de un modelo común de piedra, papel o tijera. **NB08:** despliegue del modelo con Streamlit.

## 11. Para seguir aprendiendo: ¿podemos adaptar MobileNetV2 a nuestras imágenes?

**Sí.** Hasta ahora utilizamos MobileNetV2 como un modelo que ya sabe reconocer muchos objetos. Le mostramos imágenes nuevas y observamos sus respuestas, pero **no le enseñamos categorías nuevas**.

Piensa en alguien que ya aprendió a reconocer cientos de objetos. Si ahora queremos que reconozca **las categorías del dataset que utilizaste en NB05**, no necesita aprender desde cero qué son las formas, los bordes o las texturas. Puede aprovechar ese conocimiento y aprender a distinguir tus categorías.

Eso es lo que buscamos al adaptar un **modelo preentrenado**.

### Tres ideas que no debemos confundir

- **Inferencia (lo que hicimos en NB06):** mostramos una imagen y el modelo entrega una predicción. **Sus pesos no cambian**; por eso no aprende de las imágenes que probamos.
- **Transfer Learning (aprendizaje por transferencia):** aprovechamos las características que MobileNetV2 ya aprendió y añadimos una nueva capa de clasificación para nuestras categorías. Primero entrenamos **solo esa capa nueva**; las capas originales permanecen congeladas.
- **Fine-Tuning (ajuste fino):** después de entrenar la nueva capa, podemos **descongelar algunas capas finales** de MobileNetV2 y continuar el entrenamiento con cuidado. Así modificamos una parte de sus pesos para que se adapte mejor a nuestras imágenes.

**Importante:** reemplazar la capa de clasificación y entrenarla no es todavía, por sí solo, el ajuste fino de las capas originales. Es la primera etapa del aprendizaje por transferencia. El *Fine-Tuning* comienza cuando permitimos que algunas capas preentrenadas también actualicen sus pesos.

### Ejemplo: reconocer las categorías de tu dataset del NB05

MobileNetV2 fue entrenada para clasificar **1.000 categorías de ImageNet**, no específicamente las categorías de tu dataset. Si queremos que sus respuestas correspondan a **las categorías de tus carpetas del NB05**, debemos cargar esas imágenes etiquetadas y adaptar la salida del modelo.

### ¿Qué pasos seguiríamos para hacerlo?

1. **Preparar las imágenes.** **Cada estudiante debe cargar su propio dataset**, por ejemplo, el que utilizó en NB05, con las imágenes organizadas en carpetas por categoría. Separar las imágenes en conjuntos de **entrenamiento, validación y prueba**. La prueba se reserva para comprobar el resultado final con imágenes que el modelo no haya utilizado para aprender.
2. **Cargar MobileNetV2 preentrenada.** Aprovechar sus capas para reconocer características visuales, pero cargarla **sin su capa original de clasificación de 1.000 categorías**.
3. **Agregar una nueva salida.** Incorporar una capa capaz de entregar **tantos resultados como categorías tenga tu dataset**.
4. **Entrenar primero la nueva capa.** Mantener congeladas las capas originales de MobileNetV2. En esta etapa, solo se actualizan los pesos de las capas nuevas. Este paso se conoce como **extracción de características** dentro del aprendizaje por transferencia.
5. **Realizar Fine-Tuning.** Si es necesario mejorar la adaptación, descongelar **algunas de las últimas capas** de MobileNetV2 y continuar el entrenamiento con una **tasa de aprendizaje pequeña**, para evitar alterar bruscamente lo que el modelo ya aprendió.
6. **Evaluar y comparar.** Observar las predicciones con las mismas imágenes de validación y comparar la exactitud antes y después del ajuste fino. Para una evaluación final rigurosa se necesitarían imágenes de prueba independientes. Podemos utilizar, por ejemplo, **exactitud (accuracy)** y una **matriz de confusión** para identificar qué categorías se confunden.

### ¿Siempre mejora el modelo al realizar Fine-Tuning?

**No necesariamente.** Si tenemos pocas imágenes, categorías mal etiquetadas o entrenamos demasiado, el modelo puede memorizar ejemplos y funcionar peor con imágenes nuevas. A esto se le llama **sobreajuste**. Por eso es importante comparar los resultados antes y después, en lugar de asumir que más entrenamiento siempre significa mejores predicciones.

### Relación con los notebooks del curso

En **NB05** construimos y entrenamos una CNN propia. En **NB06** utilizamos una CNN preentrenada para hacer predicciones **sin modificar sus pesos**. El aprendizaje por transferencia y el *Fine-Tuning* serían un **experimento posterior**: adaptar MobileNetV2 y comparar su desempeño con nuestra CNN.

**Esta sección es una propuesta para profundizar, no una actividad que debas ejecutar ahora.** En **NB07** continuaremos con la preparación e interoperabilidad de un modelo ya entrenado; en **NB08**, con su despliegue en Streamlit.


### Laboratorio opcional: adaptar MobileNetV2 a tu dataset

**Cada estudiante debe cargar su propio dataset**, por ejemplo, el utilizado en NB05. **No utilizaremos piedra, papel y tijera en este laboratorio**. Si tus carpetas se llaman `gatos` y `perros`, el nuevo clasificador aprenderá esas dos categorías; si son otras, el código las reconocerá automáticamente.

**Preparación (pasos 1 y 2).** Coloca las imágenes en Google Drive, con **una subcarpeta por categoría**. Ejecuta primero la sección 7, conecta Drive y ajusta `DATASET_DIR` a la ruta de **tu** carpeta. Luego ejecuta las celdas de este laboratorio en orden. Necesitas al menos dos categorías y varias imágenes por categoría; con pocas imágenes, los resultados serán inestables.

El ejercicio utiliza **entrenamiento (80 %) y validación (20 %)**. La validación permite observar cómo se comporta el modelo con imágenes que no se usan para actualizar sus pesos. Para evaluar rigurosamente un modelo terminado necesitaríamos, además, un **conjunto de prueba independiente**. Aquí la validación es una demostración didáctica, no una certificación de rendimiento.

#### Paso 3A. Leer tu dataset y dividir las imágenes

**¿Qué significan las variables iniciales?**

| Variable | Valor inicial | ¿Para qué sirve? |
|---|---|---|
| `TAMANO_IMAGEN` | `(160, 160)` | Redimensiona las imágenes a 160 × 160 píxeles. No agrega detalle a las imágenes pequeñas. |
| `LOTE` | `16` | Procesa grupos de hasta 16 imágenes antes de actualizar los pesos. Se llama *batch size*. |
| `SEMILLA` | `42` | Controla la selección aleatoria para repetir la misma división. El número 42 no tiene significado especial. |
| `validation_split` | `0.2` | Reserva aproximadamente el 20 % de las imágenes para validación; el resto se usa para entrenar. |

La función `image_dataset_from_directory` **lee los nombres de las carpetas como etiquetas**. Se ejecuta dos veces: `subset="training"` crea el conjunto de entrenamiento y `subset="validation"` el de validación. Usar la **misma semilla** asegura que ambas partes correspondan a la misma división.

**Observa la salida:** debe mostrar cuántas imágenes encontró y los nombres de **tus categorías**.

In [ ]:
# Usar la ruta del dataset propio configurada en la sección 7.
assert "DATASET_DIR" in globals() and DATASET_DIR.is_dir(), (
    "Ejecuta la sección 7 y configura DATASET_DIR con tu dataset personal."
)
TAMANO_IMAGEN = (160, 160)  # Alto y ancho de cada imagen
LOTE = 16                   # Imágenes procesadas por lote
SEMILLA = 42                # Hace reproducible la división
PORCENTAJE_VALIDACION = 0.2 # 20 % para validar; 80 % para entrenar

entrenamiento = tf.keras.utils.image_dataset_from_directory(
    str(DATASET_DIR), validation_split=PORCENTAJE_VALIDACION,
    subset="training", seed=SEMILLA,
    image_size=TAMANO_IMAGEN, batch_size=LOTE
)
validacion = tf.keras.utils.image_dataset_from_directory(
    str(DATASET_DIR), validation_split=PORCENTAJE_VALIDACION,
    subset="validation", seed=SEMILLA,
    image_size=TAMANO_IMAGEN, batch_size=LOTE
)
categorias = entrenamiento.class_names
assert len(categorias) >= 2, "Se necesitan al menos dos carpetas/categorías."
assert len(validacion.file_paths) > 0, "No hay imágenes suficientes para validar."
print("Categorías detectadas:", categorias)
print("Imágenes de entrenamiento:", len(entrenamiento.file_paths))
print("Imágenes de validación:", len(validacion.file_paths))

#### Paso 3B. ¿Dónde agregamos la nueva capa y por qué?

MobileNetV2 tiene muchas capas que aprendieron a identificar **bordes, texturas y formas**. Queremos aprovecharlas, pero no su clasificador original de 1.000 categorías ImageNet.

El recorrido de una imagen será:

**Imagen propia → MobileNetV2 sin clasificador original → GlobalAveragePooling2D → nueva capa Dense → predicción de tus categorías.**

- **`include_top=False`** elimina la parte final de clasificación de ImageNet. **No** borra las capas que extraen características.
- **`base.trainable=False`** congela inicialmente los pesos de MobileNetV2.
- **`GlobalAveragePooling2D`** resume los mapas de características en un vector compacto, adecuado para clasificar.
- **`Dense(len(categorias), activation="softmax")`** es la **nueva capa de salida**: tiene una neurona por cada categoría de tu dataset y produce probabilidades para esas categorías.

**¿Por qué va al final?** Porque primero necesitamos que MobileNetV2 extraiga características y después decidir a qué categoría corresponden. Podemos añadir otras capas entre el extractor y la salida, pero **no insertarlas en cualquier lugar sin adaptar la arquitectura y las dimensiones de los datos**. En este ejercicio mantenemos una arquitectura sencilla.

**Importante:** aquí **construimos** la nueva capa; todavía **no la entrenamos**. Eso ocurre en el paso 4.

In [ ]:
# Creamos una NUEVA instancia de MobileNetV2 para el laboratorio.
# La usada anteriormente en NB06 permanece disponible como "mobilenet".
base = tf.keras.applications.MobileNetV2(
    input_shape=(160, 160, 3), weights="imagenet", include_top=False
)
base.trainable = False  # Congelamos todas las capas preentrenadas

entrada = tf.keras.Input(shape=(160, 160, 3))
x = tf.keras.applications.mobilenet_v2.preprocess_input(entrada)
x = base(x, training=False)  # Extrae características sin actualizar estadísticas internas
x = tf.keras.layers.GlobalAveragePooling2D()(x)  # Resume las características
salida = tf.keras.layers.Dense(
    len(categorias), activation="softmax", name="clasificador_propio"
)(x)  # Una salida por categoría del dataset del estudiante
modelo_adaptado = tf.keras.Model(entrada, salida)

print("Categorías que reconocerá el nuevo clasificador:", categorias)
modelo_adaptado.summary()

#### Paso 4. Entrenar solamente la nueva capa

La nueva capa ya está incorporada. Ahora aprenderá a relacionar las características visuales extraídas por MobileNetV2 con **las categorías de tus carpetas**.

**¿Qué significa el código?**

- `compile` define **cómo aprenderá** el modelo: el optimizador `Adam`, la tasa de aprendizaje `0.001`, la función de pérdida para etiquetas enteras y la métrica `accuracy`.
- `fit` ejecuta el entrenamiento. `epochs=3` significa que el modelo recorrerá **tres veces** las imágenes de entrenamiento.
- `validation_data=validacion` mide el comportamiento con las imágenes reservadas, **sin utilizarlas para actualizar los pesos**.
- Como `base.trainable=False`, **solo se entrena el nuevo clasificador**. Esto es aprendizaje por transferencia, todavía no Fine-Tuning.

**Antes de pasar al paso 5, guardaremos una copia de los pesos.** Así podremos comparar posteriormente las predicciones del modelo con solo Transfer Learning frente al modelo con Fine-Tuning.

In [ ]:
TASA_TRANSFERENCIA = 0.001
EPOCAS_TRANSFERENCIA = 3
modelo_adaptado.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=TASA_TRANSFERENCIA),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
historial_transferencia = modelo_adaptado.fit(
    entrenamiento, validation_data=validacion,
    epochs=EPOCAS_TRANSFERENCIA
)
# Copia independiente de los pesos antes del Fine-Tuning.
pesos_transferencia = [peso.copy() for peso in modelo_adaptado.get_weights()]
print("Exactitud de validación (solo nueva capa):",
      round(historial_transferencia.history["val_accuracy"][-1], 3))

#### Paso 5. Fine-Tuning: ¿qué podemos modificar?

Hasta aquí, MobileNetV2 estaba congelada. **Ahora permitiremos actualizar algunos de sus pesos originales**, además de los del clasificador nuevo.

| Hiperparámetro | Valor de ejemplo | ¿Qué ocurre si lo cambias? |
|---|---|---|
| **Capas finales descongeladas** | `20` | Con más capas, adaptamos una mayor parte de MobileNetV2, pero aumenta el riesgo de sobreajuste. |
| **Tasa de aprendizaje** | `0.00001` | Controla el tamaño de los cambios en los pesos. En Fine-Tuning usamos un valor pequeño para conservar lo aprendido. |
| **Épocas** | `3` | Determina cuántas veces recorremos las imágenes de entrenamiento. Más épocas no garantizan mejor resultado. |
| **Tamaño de lote** | `16` (paso 3A) | Cambia cuántas imágenes se procesan por actualización. También afecta memoria y dinámica de entrenamiento. |
| **Optimizador** | `Adam` | Determina cómo se calculan las actualizaciones de los pesos. |
| **Resolución de entrada** | `160 × 160` (paso 3A) | Cambiarla requiere recrear el modelo con una entrada compatible. Puede afectar detalle y consumo de recursos. |

**Pesos e hiperparámetros no son lo mismo:** los **pesos** son valores aprendidos por la red; los **hiperparámetros** son decisiones que nosotros configuramos para controlar el entrenamiento.

**Lee estas instrucciones del código:**
- `base.trainable=True`: habilita el entrenamiento de la base.
- `base.layers[:-CAPAS_A_DESCONGELAR]`: mantiene congeladas las capas anteriores a las últimas seleccionadas.
- Las capas `BatchNormalization` continúan congeladas para no alterar sus estadísticas internas durante este ajuste.
- Después de cambiar qué capas pueden aprender, es necesario volver a ejecutar `compile`.

**Experimento sugerido:** comienza con los valores indicados. Si decides experimentar, modifica **un hiperparámetro a la vez**, vuelve a ejecutar los pasos correspondientes y compara los resultados. No es obligatorio conseguir una mejora.

In [ ]:
CAPAS_A_DESCONGELAR = 20
TASA_FINE_TUNING = 0.00001
EPOCAS_FINE_TUNING = 3

base.trainable = True
# Congelamos todas las capas anteriores a las últimas 20.
for capa in base.layers[:-CAPAS_A_DESCONGELAR]:
    capa.trainable = False
# Evitamos actualizar BatchNormalization durante el ajuste fino.
for capa in base.layers[-CAPAS_A_DESCONGELAR:]:
    if isinstance(capa, tf.keras.layers.BatchNormalization):
        capa.trainable = False

# Volvemos a compilar porque cambiamos las capas entrenables.
modelo_adaptado.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=TASA_FINE_TUNING),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
historial_ajuste = modelo_adaptado.fit(
    entrenamiento, validation_data=validacion,
    epochs=EPOCAS_FINE_TUNING
)
print("Exactitud de validación (con Fine-Tuning):",
      round(historial_ajuste.history["val_accuracy"][-1], 3))

#### Paso 6A. Comparar la exactitud antes y después

La exactitud (*accuracy*) es la proporción de predicciones correctas. Compararemos **la misma tarea y las mismas categorías**: el modelo después de entrenar el clasificador nuevo frente al modelo después de Fine-Tuning.

**Atención:** la exactitud de validación no equivale a una evaluación final con imágenes de prueba independientes. Con datasets pequeños puede variar mucho.

In [ ]:
exactitud_antes = historial_transferencia.history["val_accuracy"][-1]
exactitud_despues = historial_ajuste.history["val_accuracy"][-1]
comparacion = pd.DataFrame({
    "Etapa": ["Transfer Learning: solo clasificador", "Fine-Tuning: capas finales"],
    "Exactitud de validación": [exactitud_antes, exactitud_despues]
})
display(comparacion)
print("Cambio en exactitud:", f"{exactitud_despues - exactitud_antes:+.2%}")

#### Paso 6B. ¡Ver las predicciones de los tres modelos con las mismas imágenes!

Comparar solo números puede resultar abstracto. Ahora seleccionaremos **hasta cinco imágenes del conjunto de validación** y observaremos qué predice cada versión:

1. **MobileNetV2 original:** la misma red usada en las actividades anteriores del NB06. Predice entre **1.000 categorías ImageNet**, que pueden no coincidir con las de tu dataset.
2. **Transfer Learning:** MobileNetV2 con **tu clasificador nuevo entrenado**, pero sin modificar los pesos de la base preentrenada.
3. **Fine-Tuning:** el mismo clasificador después de ajustar también algunas capas finales de MobileNetV2.

**Las tres versiones reciben exactamente las mismas imágenes.** Para comparar las etapas 2 y 3, reconstruiremos temporalmente las predicciones con la copia de pesos que guardamos en el paso 4 y luego restauraremos los pesos del Fine-Tuning.

**Cómo leer la tabla:** la etiqueta real es el nombre de la carpeta de la imagen. Las predicciones de Transfer Learning y Fine-Tuning utilizan **tus categorías** y pueden compararse directamente. La predicción original utiliza **categorías ImageNet**, por lo que sus probabilidades **no son directamente comparables** con las de los modelos adaptados.

Usamos imágenes de validación como demostración visual. Si quisieras evaluar formalmente la calidad final del modelo, deberías usar imágenes de prueba independientes.

In [ ]:
# Tomamos hasta cinco archivos de validación (no usados para actualizar los pesos).
rutas_ejemplo = validacion.file_paths[:5]
assert rutas_ejemplo, "No hay imágenes de validación para comparar."

# La MobileNetV2 original ya se cargó en la sección 4.
assert "mobilenet" in globals(), "Ejecuta antes la sección 4 para cargar MobileNetV2 original."

def preparar_imagen(ruta, tamano):
    with Image.open(ruta) as im:
        imagen = np.asarray(im.convert("RGB").resize(tamano), dtype=np.float32)
    return imagen[None, ...]

def predecir_adaptado(ruta):
    imagen = preparar_imagen(ruta, TAMANO_IMAGEN)
    probabilidades = modelo_adaptado.predict(imagen, verbose=0)[0]
    indice = int(np.argmax(probabilidades))
    return categorias[indice], float(probabilidades[indice])

# Guardamos los pesos del Fine-Tuning para restaurarlos después.
pesos_fine_tuning = [peso.copy() for peso in modelo_adaptado.get_weights()]
filas = []
try:
    for ruta in rutas_ejemplo:
        # Modelo original: clasificador ImageNet de 1.000 categorías.
        imagen_original = preparar_imagen(ruta, (224, 224))
        imagen_original = tf.keras.applications.mobilenet_v2.preprocess_input(imagen_original)
        probabilidades_original = mobilenet.predict(imagen_original, verbose=0)
        top_original = tf.keras.applications.mobilenet_v2.decode_predictions(
            probabilidades_original, top=1
        )[0][0]

        # Transfer Learning: recuperamos la versión previa al Fine-Tuning.
        modelo_adaptado.set_weights(pesos_transferencia)
        etiqueta_transferencia, confianza_transferencia = predecir_adaptado(ruta)

        # Fine-Tuning: restauramos la versión ajustada.
        modelo_adaptado.set_weights(pesos_fine_tuning)
        etiqueta_ajuste, confianza_ajuste = predecir_adaptado(ruta)

        filas.append({
            "Imagen": Path(ruta).name,
            "Categoría real": Path(ruta).parent.name,
            "Original (ImageNet)": top_original[1],
            "Transfer Learning": etiqueta_transferencia,
            "Fine-Tuning": etiqueta_ajuste,
            "¿Acertó TL?": etiqueta_transferencia == Path(ruta).parent.name,
            "¿Acertó FT?": etiqueta_ajuste == Path(ruta).parent.name,
        })
finally:
    # Dejamos activo el modelo con Fine-Tuning.
    modelo_adaptado.set_weights(pesos_fine_tuning)

display(pd.DataFrame(filas))
print("Original = categorías ImageNet; TL y FT = categorías de tu dataset.")

#### Paso 6C. Observar las imágenes y discutir los resultados

La tabla anterior muestra las etiquetas predichas. Ahora verás **las mismas imágenes** para interpretar visualmente por qué el modelo acertó o se equivocó.

In [ ]:
import matplotlib.pyplot as plt
fig, ejes = plt.subplots(1, len(rutas_ejemplo), figsize=(3 * len(rutas_ejemplo), 3.5))
if len(rutas_ejemplo) == 1:
    ejes = [ejes]
for eje, ruta in zip(ejes, rutas_ejemplo):
    with Image.open(ruta) as im:
        eje.imshow(im.convert("RGB"))
    eje.set_title(Path(ruta).parent.name + "\n" + Path(ruta).name, fontsize=9)
    eje.axis("off")
plt.tight_layout()
plt.show()

**Preguntas para reflexionar:** ¿qué imágenes cambian de predicción entre Transfer Learning y Fine-Tuning? ¿Los cambios fueron correctos? ¿Qué diferencias observas respecto de la predicción ImageNet original? ¿Por qué una mayor confianza no garantiza una respuesta correcta?

**Este laboratorio es opcional.** No cambia las actividades anteriores del NB06. En NB07 continuaremos con interoperabilidad y ONNX, y en NB08 con Streamlit.